# TRAIL on a Colab GPU

Runs the full TRAIL study on a CUDA GPU: synthetic pretraining, the 100-well holdout, six ablations and five-fold CV.
Every step writes to Google Drive and is skipped when its output already exists, so a disconnected session resumes where it stopped.

Runtime → Change runtime type → GPU, then run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code

`SOURCE = "github"` clones the `feat/attention-model` branch; `"zip"` asks you to upload a zip of the repository instead.

In [ ]:
SOURCE = "github"
BRANCH = "feat/attention-model"
import os, subprocess
os.chdir("/content")
if SOURCE == "github":
    if not os.path.isdir("repo"):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/gun-py/Kaggle_ROGII_2026", "repo"], check=True)
else:
    from google.colab import files
    name = next(iter(files.upload()))
    subprocess.run(["unzip", "-q", "-o", name, "-d", "repo"], check=True)
%cd /content/repo
!git log --oneline -1 2>/dev/null || true
!pip install -q -e ".[seq]"


## Data

`DATA = "parts"` takes the competition CSVs as numbered pieces of `rogii_data.zip` (made with `split -b 9000000 -d -a 2`), one upload per piece, and checks the reassembled zip against its SHA-256. `"zip"` takes the whole zip in one upload; `"kaggle"` downloads the data with your own `kaggle.json`.

In [ ]:
DATA = "parts"
PARTS, SHA256 = 22, "790dc9d9e379e5b8a17b4bd3a3e9462b2984dc5b2c73328743b752b011a7da98"
import os, pathlib, subprocess
if not pathlib.Path("data/raw/sample_submission.csv").exists():
    from google.colab import files
    os.makedirs("data/raw", exist_ok=True)
    if DATA == "parts":
        import hashlib
        while len([p for p in os.listdir() if p.startswith("rogii_data.zip.part")]) < PARTS:
            files.upload()
        subprocess.run("cat rogii_data.zip.part* > rogii_data.zip && rm rogii_data.zip.part*", shell=True, check=True)
        assert hashlib.sha256(open("rogii_data.zip", "rb").read()).hexdigest() == SHA256, "reassembled zip does not match"
        name, DATA = "rogii_data.zip", "zip"
    else:
        name = next(iter(files.upload()))
    if DATA == "zip":
        subprocess.run(["unzip", "-q", "-o", name, "-d", "data/raw"], check=True)
        os.remove(name)
    else:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.replace(name, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        subprocess.run(["kaggle", "competitions", "download", "-c", "rogii-wellbore-geology-prediction", "-p", "data/raw", "-q"], check=True)
        subprocess.run(["unzip", "-q", "-o", "data/raw/rogii-wellbore-geology-prediction.zip", "-d", "data/raw"], check=True)
!ls data/raw; ls data/raw/train | wc -l

## Output on Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/trail_runs"
os.makedirs(f"{OUT}/logs", exist_ok=True)
print(OUT)

## The study

Same configuration as the repository (`src/rogii/configs/attention/trail.toml`): up to 15 fine-tuning epochs, early stopping after 4.

In [ ]:
import subprocess, time, pathlib
def step(name, args, done):
    if pathlib.Path(done).exists():
        print(f"skip {name}: {done} exists"); return
    t0 = time.time()
    with open(f"{OUT}/logs/{name}.log", "w") as log:
        r = subprocess.run(["rogii", "attn", *args, "--data", "data/raw", "--out", OUT], stdout=log, stderr=subprocess.STDOUT)
    tail = pathlib.Path(f"{OUT}/logs/{name}.log").read_text().strip().splitlines()[-1:]
    print(f"{name}: exit {r.returncode}, {time.time() - t0:.0f}s", tail)
    if r.returncode:
        raise RuntimeError(f"{name} failed; see {OUT}/logs/{name}.log")

INIT = ["--init", f"{OUT}/pretrain.pt"]
step("pretrain", ["pretrain"], f"{OUT}/pretrain.pt")
step("holdout_full", ["holdout", "--ablation", "full", *INIT], f"{OUT}/holdout/trail_full_summary.json")
step("holdout_no_pretrain", ["holdout", "--ablation", "no_pretrain"], f"{OUT}/holdout/trail_no_pretrain_summary.json")
for ab in ["no_neighbours", "no_typewell", "single_pass", "no_offset_bias", "integrated"]:
    step(f"holdout_{ab}", ["holdout", "--ablation", ab, *INIT], f"{OUT}/holdout/trail_{ab}_summary.json")
step("cv", ["cv", *INIT], f"{OUT}/cv/trail_cv4.pt")

## Results

Prints the summaries, then zips everything except the cached well set. Download `trail_runs.zip` and put it in `artifacts/` of your local checkout.

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{OUT}/holdout/*_summary.json")):
    s = json.load(open(p)); print(f"{s['method']:<22} {s['rmse']:.3f} ft  (carry {s['carry']:.3f})")
!cd /content/drive/MyDrive && zip -q -r /content/trail_runs.zip trail_runs -x "trail_runs/wellset.pkl"
from google.colab import files
files.download("/content/trail_runs.zip")